In [11]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.agents import create_agent
import os

In [12]:
# import getpass
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", None)

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "kimi-k2:1t-cloud")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")
OLLAMA_MODEL_URL = os.getenv("OLLAMA_MODEL_URL", "http://localhost:11434")

print(f"LANGSMITH_TRACING: {LANGSMITH_TRACING}")
print(f"LANGSMITH_API_KEY: {LANGSMITH_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL: {OLLAMA_MODEL}")
print(f"OLLAMA_API_KEY: {OLLAMA_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL_URL: {OLLAMA_MODEL_URL}")

LANGSMITH_TRACING: True
LANGSMITH_API_KEY: REDACTED_LANGSMITH_TOKEN
OLLAMA_MODEL: kimi-k2:1t-cloud
OLLAMA_API_KEY: ollama
OLLAMA_MODEL_URL: http://localhost:11434


In [13]:
def mock_llm(state: MessagesState):
    print("mock_llm called with state:", state)
    return {"messages": [{"role": "ai", "content": "hello world"}]}

def mock_llm2(state: MessagesState):
    print("mock_llm2 called with state:", state)
    return {"messages": [{"role": "ai", "content": "hello again"}]}


graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_node(mock_llm2)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", "mock_llm2")
graph.add_edge("mock_llm2", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})


mock_llm called with state: {'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='0cc1d4ad-9d8e-47ab-a188-e70ed3d406ee')]}
mock_llm2 called with state: {'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='0cc1d4ad-9d8e-47ab-a188-e70ed3d406ee'), AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='03a19f01-38e2-4077-9655-aca253c2c000', tool_calls=[], invalid_tool_calls=[])]}


{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='0cc1d4ad-9d8e-47ab-a188-e70ed3d406ee'),
  AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='03a19f01-38e2-4077-9655-aca253c2c000', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='hello again', additional_kwargs={}, response_metadata={}, id='150c012e-5566-49be-b487-698288073ec7', tool_calls=[], invalid_tool_calls=[])]}

In [14]:
from langchain.chat_models import init_chat_model

print(f"OLLAMA_MODEL: {OLLAMA_MODEL}")
print(f"OLLAMA_API_KEY: {OLLAMA_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL_URL: {OLLAMA_MODEL_URL}")

model = init_chat_model(
  OLLAMA_MODEL,
  model_provider="openai",
  api_key=OLLAMA_API_KEY,
  base_url=f"{OLLAMA_MODEL_URL}/v1",
)

OLLAMA_MODEL: kimi-k2:1t-cloud
OLLAMA_API_KEY: ollama
OLLAMA_MODEL_URL: http://localhost:11434


In [15]:
from langchain.tools import tool
from datetime import date, timedelta

@tool
def get_current_date() -> str:
    """Get today's date, day of the week, and nearby reference dates.

    Use this FIRST before scheduling anything so you can resolve
    relative expressions like 'next Tuesday', 'this Friday', 'in 3 days', etc.
    """
    today = date.today()
    weekday = today.strftime("%A")
    # Calculate next occurrence of each weekday
    days_of_week = {}
    for i in range(1, 8):
        d = today + timedelta(days=i)
        days_of_week[d.strftime("%A")] = d.isoformat()
    upcoming = ", ".join(f"{day}: {dt}" for day, dt in days_of_week.items())
    return (
        f"Today is {weekday}, {today.isoformat()}. "
        f"Upcoming days: {upcoming}"
    )

@tool
def create_calendar_event(
  title: str,
  start_time: str,
  end_time: str,
  attendees: list[str],
  location: str
) -> str:
  """Create a calendar event. Requires exact ISO datetime format."""
  # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
  return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

@tool
def send_email(
  to: list[str],  # email addresses
  subject: str,
  body: str,
  cc: list[str] = []
) -> str:
  """Send an email via email API. Requires properly formatted addresses."""
  # Stub: In practice, this would call SendGrid, Gmail API, etc.
  return f"Email sent to {', '.join(to)} - Subject: {subject}"

@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

In [16]:
from langchain.agents.middleware import HumanInTheLoopMiddleware 

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. Follow these steps IN ORDER:\n"
    "1. ALWAYS call get_current_date first to resolve relative dates like 'next Tuesday'.\n"
    "2. ALWAYS call get_available_time_slots to check if the requested time is available.\n"
    "3. If the requested time is NOT in the available slots, DO NOT create the event. "
    "Instead, suggest the closest available time slot and ask for confirmation.\n"
    "4. Only call create_calendar_event once the time is confirmed as available.\n"
    "5. Always confirm what was scheduled in your final response.\n\n"
    "IMPORTANT: Never skip the availability check. Never create an event at an unavailable time."
)

calendar_agent = create_agent(
    model,
    tools=[get_current_date, create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    # middleware=[
    #     HumanInTheLoopMiddleware(
    #         interrupt_on={"create_calendar_event": True},
    #         description_prefix="Calendar event pending approval",
    #     )
    # ],
)


EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
    # middleware=[ 
    #     HumanInTheLoopMiddleware( 
    #         interrupt_on={"send_email": True}, 
    #         description_prefix="Outbound email pending approval", 
    #     ), 
    # ],
)

In [17]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm ET')
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].text

In [18]:
from langgraph.checkpoint.memory import InMemorySaver 

SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. Follow these rules:\n"
    "1. ALWAYS schedule the calendar event FIRST using schedule_event.\n"
    "2. Wait for the scheduling result before sending any emails.\n"
    "3. Only send an email AFTER the event is successfully scheduled.\n"
    "4. Include the confirmed meeting details (date, time) in the email.\n"
    "5. If the scheduling suggests an alternative time, report that to the user "
    "before sending any email.\n"
    "When a request involves multiple actions, execute them in sequence, not in parallel."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),
)

In [19]:
query = (
    "Next Tuesday at 9am ET, schedule a meeting with the design team "
    "Ana and Bob, and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "6"}}

interrupts = []
for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\nINTERRUPTED: {interrupt_.id}")

================================== Ai Message ==================================

I'll help you schedule the meeting and then send the email reminder to the design team. Let me start by scheduling the meeting for next Tuesday at 9am ET.
Tool Calls:
  schedule_event (functions.schedule_event:0)
 Call ID: functions.schedule_event:0
  Args:
    request: meeting with design team Ana and Bob next Tuesday at 9am ET to review the new mockups
================================= Tool Message =================================
Name: schedule_event

Great! I've successfully scheduled your design team meeting with Ana and Bob. Here are the details:

**Meeting Scheduled:**
- **Title:** Design Team Meeting - Review New Mockups
- **Date:** Tuesday, March 3rd, 2026
- **Time:** 9:00 AM - 10:00 AM ET
- **Attendees:** Ana and Bob
- **Location:** TBD

The meeting has been created and both Ana and Bob are available at that time. You may want to send calendar invites to confirm the meeting details and agree on

In [20]:
for interrupt_ in interrupts:
    for request in interrupt_.value["action_requests"]:
        print(f"INTERRUPTED: {interrupt_.id}")
        print(f"{request['description']}\n")